# Image Editing

**Module:** 17 — Image Generation

Inpaint, outpaint, img2img, instruction edits, and identity-preserving workflows.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Distinguish edit modes and failure modes
- Design masks and strength schedules
- Apply identity-preservation for faces/products
- Compose multi-step edit graphs with QA gates


## Edit Modes

### Definition
Editing modifies a canvas: **img2img**, **inpaint**, **outpaint**, **instruction edit**, **variation**.

### Why it matters
Most production value is editing, not blank-noise generation.

### How it works
Encode init → noise per strength → denoise with prompt/controls → composite with feathered mask → QA.

### Intuition
Photo retouching with layers and masks — generative edition.

### Pitfalls
- Hard mask edges → seams
- Strength too high → identity wipe
- Prompt fights the init image

### When to use
Catalog cleanup, ad variants, creative iteration.


| Mode | Changes | Lock trick |
|------|---------|------------|
| img2img | Whole image | Low strength |
| inpaint | Masked pixels | Feather; local prompt |
| outpaint | Border pixels | Soft overlap |
| instruction | Semantic delta | Clear verbs |
| upscale+refine | Detail | Tile consistency |

```mermaid
flowchart TD
  I[Init] --> M{Mode}
  M -->|img2img| G[Global denoise]
  M -->|inpaint| K[Masked denoise]
  M -->|outpaint| X[Expanded canvas]
  G --> Q[QA]
  K --> Q
  X --> Q
```


In [ ]:
# Demo 1: mask + strength risk
from dataclasses import dataclass

@dataclass
class EditJob:
    mode: str
    strength: float
    mask_coverage: float
    prompt: str

    def risk(self) -> str:
        if self.mode == "img2img" and self.strength > 0.7: return "high_identity_loss"
        if self.mode == "inpaint" and self.mask_coverage > 0.5 and self.strength > 0.65: return "high_scene_rewrite"
        return "ok"

for j in [EditJob("img2img", 0.35, 1.0, "warm grade"), EditJob("inpaint", 0.55, 0.08, "teal mug"),
          EditJob("img2img", 0.85, 1.0, "cyberpunk")]:
    print(j.mode, j.strength, j.risk())


In [ ]:
# Demo 2: feathered composite (1D analogy)
def composite(base, edit, mask):
    return [b * (1 - m) + e * m for b, e, m in zip(base, edit, mask)]

base, edit = [0.1]*5, [0.9]*5
print("hard", [round(x, 2) for x in composite(base, edit, [0,0,1,1,0])])
print("soft", [round(x, 2) for x in composite(base, edit, [0,0.25,1,0.25,0])])


## Identity Preservation

### Definition
Keep faces/logos/SKUs recognizable via low strength, masks, reference adapters, and similarity floors.

### Why it matters
Brand and trust die when the product morphs.

### How it works
Freeze unmasked pixels; constrain denoise; reject below similarity floor.

### Intuition
Surgery with a strict 'still looks like the client' checklist.

### Pitfalls
- Global restyle on identity crops
- No automated similarity gate

### When to use
People (policy!), mascots, product catalog shots.


In [ ]:
# Demo 3: cosine identity gate
import math

def cos(a, b):
    dot = sum(x*y for x,y in zip(a,b))
    na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(y*y for y in b))
    return dot / (na * nb + 1e-9)

def accept(before, after, floor=0.85):
    s = cos(before, after)
    return s >= floor, round(s, 3)

face = [0.2, 0.8, 0.1, 0.4]
print(accept(face, [0.21, 0.79, 0.12, 0.39]))
print(accept(face, [0.9, 0.1, 0.7, 0.2]))


In [ ]:
# Demo 4: instruction-edit request shape
ANTHROPIC_API_KEY = "YOUR_ANTHROPIC_API_KEY"
edit_req = {
    "image_url": "https://example.invalid/product.png",
    "instruction": "Remove the cardboard box; keep headphones fixed; match shadows",
    "mask_url": "https://example.invalid/box_mask.png",
    "seed": 99,
}
edit_res = {"status": "ok", "image_b64": "<...>", "qa": {"identity": 0.91, "seam": 0.05}}
print(edit_req["instruction"]); print(edit_res["qa"])


### Identity tips
- Prefer inpaint over full img2img for local changes
- Feather masks; keep lighting words consistent
- Lock seeds when comparing instruction variants
- Store before/after embeddings for audit


### Try it yourself — Editing

1. Implement a seam metric along the mask boundary band.
2. Design a 3-step graph: remove → outpaint 16:9 → warm grade.
3. Define reject thresholds for identity and seam.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `strength` | How far to push from init image |
| `feathering` | Soft mask edges to hide seams |
| `instruction edit` | Language-specified image change |
| `identity floor` | Minimum similarity to accept |


### Workshop — Parameter journal — Image Editing

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Editing
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Editing

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Editing
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Editing

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Editing
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Editing

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Editing
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Editing

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Editing
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Editing

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Editing
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Image Editing

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Image Editing
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Image Editing

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Image Editing
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Editing is the production workhorse
- Mask + strength discipline preserves identity
- Automate QA with similarity and seam checks
